# Agente de Análise de Telemetria de Corrida

**Participantes:** Daniel Limaverde da Silva

## Relatório breve

`Versão do notebook para rodar no Google Colab`

Este trabalho implementa um agente capaz de analisar telemetria de uma sessão de corrida de simulação coletada do simulador "Le Mans Ultimate".

A tarefa é verificável porque cada resposta pode ser comparada com valores objetivos extraídos da própria telemetria: voltas válidas, diferença total de tempo, setor de maior perda e métricas médias do setor analisado.

O agente usa o módulo `agentkit` próprio da disciplina e decide dinamicamente quais ferramentas chamar.
A solução inclui três ferramentas próprias decoradas com `@tool`: `list_laps`, `compare_laps` e `analyze_section`.
Alguns testes exigem múltiplas etapas, por exemplo listar voltas, escolher uma referência, comparar voltas e depois analisar o setor de maior perda.

Os mecanismos adicionais usados são:

- **Estado:** `AnalysisState` registra voltas, comparação, setores analisados, evidências e observações.
- **Saída estruturada:** `StructuredTestResult`, com Pydantic, valida e padroniza o resultado de cada caso de teste, incluindo entrada, esperado, ferramentas usadas, estado observado e aprovação.

Quando o notebook é executado fora do repositório, a primeira célula de código clona o projeto do GitHub com `git clone`, instala as dependências mínimas de `requirements.txt` no ambiente Python ativo e adiciona a raiz do projeto ao `sys.path`, permitindo que os módulos locais sejam importados sem intervenção manual.




In [8]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/Daniel-Lmv/Agent_Telemetry_Analyser.git"
REPO_DIR_NAME = "Agent_Telemetry_Analyser"


def find_project_root(start: Path) -> Path | None:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "agentkit").exists() and (candidate / "data" / "sessao.csv").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    clone_target = Path.cwd() / REPO_DIR_NAME

    if not clone_target.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(clone_target)],
            check=True,
        )

    PROJECT_ROOT = find_project_root(clone_target)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Não foi possível localizar ou clonar o projeto. "
        "Verifique o acesso ao repositório."
    )

requirements_file = PROJECT_ROOT / "requirements.txt"
if requirements_file.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
        check=True,
    )

sys.path.insert(0, str(PROJECT_ROOT))

from agent.agent import create_agent, run_analysis
from agent.state import AnalysisState
from agentkit import LLMAPI
from core.preprocessing import preprocess_telemetry
from core.telemetry import Telemetry
from tools.comparison_tools import create_comparison_tools
from tools.lap_tools import create_lap_tools
from tools.section_tools import create_section_tools

print(f"Projeto carregado de: {PROJECT_ROOT}")

Projeto carregado de: /content/Agent_Telemetry_Analyser


## Carregamento e pré-processamento da telemetria

A sessão é dividida em voltas, cada volta é classificada e apenas voltas válidas devem ser usadas em comparações de desempenho.

In [9]:
telemetry = Telemetry.from_file(PROJECT_ROOT / "data" / "sessao.csv")
laps, track_length = preprocess_telemetry(telemetry.data)

lap_summary = [
    {
        "lap": lap.number,
        "status": lap.status.value,
        "duration": round(lap.duration, 3),
        "distance": round(lap.distance_covered, 1),
        "average_speed": round(lap.average_speed, 2),
        "stopped_ratio": round(lap.stopped_ratio, 3),
    }
    for lap in laps
]

print(f"Comprimento estimado da pista: {track_length:.3f} m")
lap_summary

Comprimento estimado da pista: 4896.266 m


[{'lap': 1,
  'status': 'incomplete',
  'duration': 101.19,
  'distance': 4562.8,
  'average_speed': 46.42,
  'stopped_ratio': 0.271},
 {'lap': 2,
  'status': 'valid',
  'duration': 119.59,
  'distance': 4896.3,
  'average_speed': 155.6,
  'stopped_ratio': 0.0},
 {'lap': 3,
  'status': 'valid',
  'duration': 108.69,
  'distance': 4891.5,
  'average_speed': 161.14,
  'stopped_ratio': 0.0},
 {'lap': 4,
  'status': 'valid',
  'duration': 110.79,
  'distance': 4896.5,
  'average_speed': 159.12,
  'stopped_ratio': 0.0},
 {'lap': 5,
  'status': 'valid',
  'duration': 108.29,
  'distance': 4892.7,
  'average_speed': 162.1,
  'stopped_ratio': 0.0},
 {'lap': 6,
  'status': 'valid',
  'duration': 106.89,
  'distance': 4894.7,
  'average_speed': 163.81,
  'stopped_ratio': 0.0},
 {'lap': 7,
  'status': 'valid',
  'duration': 109.19,
  'distance': 4892.5,
  'average_speed': 160.61,
  'stopped_ratio': 0.0},
 {'lap': 8,
  'status': 'valid',
  'duration': 109.79,
  'distance': 4894.8,
  'average_speed

## Modelo e ferramentas

O notebook prepara automaticamente o Ollama e o modelo necessário.

In [19]:
# Preparar Ollama e modelo no Google Colab

import os
import time
import subprocess
import urllib.request

MODEL = "qwen3.5:4b"
OLLAMA_HOST = "http://127.0.0.1:11434"

# Instala dependências e Ollama
subprocess.run(
    "apt-get update -qq && "
    "apt-get install -y -qq pciutils lshw zstd && "
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=True,
)

# Configuração usada pelo Ollama para encontrar as bibliotecas NVIDIA no Colab
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
)

# Inicia o servidor Ollama
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Aguarda o servidor ficar disponível
for _ in range(30):
    try:
        urllib.request.urlopen(
            f"{OLLAMA_HOST}/api/tags",
            timeout=2,
        )
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("O servidor Ollama não iniciou corretamente.")

print("Ollama iniciado.")

# Baixa o modelo caso ainda não esteja disponível
subprocess.run(
    ["ollama", "pull", MODEL],
    check=True,
)

print(f"Modelo {MODEL} disponível.")

Ollama iniciado.
Modelo qwen3.5:4b disponível.


In [14]:
MODEL = os.environ.get("TELEMETRY_AGENT_MODEL", "qwen3.5:4b")
BASE_URL = os.environ.get("TELEMETRY_AGENT_BASE_URL", "http://localhost:11434/v1")

llm = LLMAPI(
    model=MODEL,
    api_key=os.environ.get("TELEMETRY_AGENT_API_KEY", "ollama"),
    base_url=BASE_URL,
    temperature=0.0,
    max_tokens=4096,
)

def build_agent_and_state():
    state = AnalysisState()
    tools = (
        create_lap_tools(laps)
        + create_comparison_tools(laps, state)
        + create_section_tools(laps, state)
    )
    return create_agent(llm, tools, max_steps=10), state

print(f"Modelo: {MODEL}")
print(f"Endpoint: {BASE_URL}")

Modelo: qwen3.5:4b
Endpoint: http://localhost:11434/v1


## Casos de teste

Os dez testes abaixo cobrem consultas simples, comparações, análise de setores, descoberta dinâmica de referência, entradas inválidas e casos que exigem múltiplas chamadas de ferramenta. Os testes 1, 4 e 7 são os mais desafiadores por dependerem de mais de uma etapa de decisão do agente.

In [15]:
tests = [
    {
        "id": 1,
        "name": "Análise completa",
        "input": "Por que a volta 7 perdeu tempo em relação à volta 5? Identifique onde ocorreu a perda e analise as evidências.",
        "expected": "Usar compare_laps e analyze_section; target=7, reference=5, setor principal=2.",
        "required_tools": ["compare_laps", "analyze_section"],
        "target_lap": 7,
        "reference_lap": 5,
        "section": 2,
    },
    {
        "id": 2,
        "name": "Análise de setor",
        "input": "Analise o setor 2 da volta 7 comparando com a volta 5.",
        "expected": "Usar analyze_section; target=7, reference=5, setor=2.",
        "required_tools": ["analyze_section"],
        "target_lap": 7,
        "reference_lap": 5,
        "section": 2,
    },
    {
        "id": 3,
        "name": "Comparação de voltas",
        "input": "Compare a volta 7 com a volta 5 e informe a diferença de tempo.",
        "expected": "Usar compare_laps; target=7, reference=5.",
        "required_tools": ["compare_laps"],
        "target_lap": 7,
        "reference_lap": 5,
    },
    {
        "id": 4,
        "name": "Descoberta da referência",
        "input": "Analise a volta 7 em relação à melhor volta válida da sessão e identifique onde ocorreu a maior perda de tempo.",
        "expected": "Usar list_laps, compare_laps e analyze_section; target=7.",
        "required_tools": ["list_laps", "compare_laps", "analyze_section"],
        "target_lap": 7,
    },
    {
        "id": 5,
        "name": "Consulta da sessão",
        "input": "Quais são as voltas válidas disponíveis para análise nesta sessão?",
        "expected": "Usar list_laps e listar voltas válidas.",
        "required_tools": ["list_laps"],
    },
    {
        "id": 6,
        "name": "Outro setor",
        "input": "Analise o setor 4 da volta 7 em relação à volta 5.",
        "expected": "Usar analyze_section; target=7, reference=5, setor=4.",
        "required_tools": ["analyze_section"],
        "target_lap": 7,
        "reference_lap": 5,
        "section": 4,
    },
    {
        "id": 7,
        "name": "Referência sem volta definida",
        "input": "Encontre uma volta válida para usar como referência para a volta 8 e compare as duas voltas.",
        "expected": "Usar list_laps e compare_laps; target=8.",
        "required_tools": ["list_laps", "compare_laps"],
        "target_lap": 8,
    },
    {
        "id": 8,
        "name": "Volta inválida",
        "input": "Compare a volta 1 com a volta 5.",
        "expected": "Informar que a volta 1 não é válida.",
        "required_tools": ["compare_laps"],
        "expected_error": "not valid",
    },
    {
        "id": 9,
        "name": "Volta incompleta",
        "input": "Compare a volta 12 com a volta 5.",
        "expected": "Informar que a volta 12 não é válida.",
        "required_tools": ["compare_laps"],
        "expected_error": "not valid",
    },
    {
        "id": 10,
        "name": "Setor inválido",
        "input": "Analise o setor 6 da volta 7 comparando com a volta 5.",
        "expected": "Informar que o setor deve estar entre 1 e 5.",
        "required_tools": ["analyze_section"],
        "expected_error": "between 1 and 5",
    },
]

In [16]:
from pydantic import BaseModel, Field


class StructuredTestResult(BaseModel):
    """Resultado estruturado de um caso de teste do agente."""

    test_id: int = Field(description="Identificador numérico do teste.")
    name: str = Field(description="Nome curto do caso de teste.")
    input: str = Field(description="Entrada enviada ao agente.")
    expected: str = Field(description="Resultado esperado para o teste.")
    used_tools: list[str] = Field(description="Ferramentas chamadas pelo modelo durante a execução.")
    observed_state: dict = Field(description="Resumo do estado compartilhado após a execução.")
    final_answer: str = Field(description="Resposta final textual produzida pelo agente.")
    passed: bool = Field(description="Indica se o teste foi aprovado.")
    validation_notes: list[str] = Field(description="Motivos de reprovação ou lista vazia quando o teste passou.")


def extract_used_tools(history):
    used_tools = []
    for message in history:
        for call in message.get("tool_calls", []):
            used_tools.append(call["name"])
    return used_tools


def final_answer(history):
    for message in reversed(history):
        if message.get("role") == "assistant" and not message.get("tool_calls"):
            return message.get("content", "")
    return ""


def test_passed(test, state, history, used_tools):
    passed = True
    reasons = []

    for required_tool in test.get("required_tools", []):
        if required_tool not in used_tools:
            passed = False
            reasons.append(f"ferramenta ausente: {required_tool}")

    if "target_lap" in test and state.target_lap != test["target_lap"]:
        passed = False
        reasons.append(f"target_lap esperado={test['target_lap']} obtido={state.target_lap}")

    if "reference_lap" in test and state.reference_lap != test["reference_lap"]:
        passed = False
        reasons.append(f"reference_lap esperado={test['reference_lap']} obtido={state.reference_lap}")

    if "section" in test and test["section"] not in state.analyzed_sections:
        passed = False
        reasons.append(f"setor não analisado: {test['section']}")

    if "expected_error" in test:
        needle = test["expected_error"].lower()
        found = any(needle in str(message.get("content", "")).lower() for message in history)
        if not found:
            passed = False
            reasons.append(f"erro esperado não encontrado: {test['expected_error']}")

    return passed, reasons


In [17]:
results = []
structured_results = []

for test in tests:
    agent, state = build_agent_and_state()
    history = run_analysis(agent, test["input"])
    used_tools = extract_used_tools(history)
    passed, reasons = test_passed(test, state, history, used_tools)
    obtained = {
        "used_tools": used_tools,
        "state": state.summary(),
        "final_answer": final_answer(history),
        "reasons": reasons,
    }
    structured_result = StructuredTestResult(
        test_id=test["id"],
        name=test["name"],
        input=test["input"],
        expected=test["expected"],
        used_tools=used_tools,
        observed_state=state.summary(),
        final_answer=final_answer(history),
        passed=passed,
        validation_notes=reasons,
    )
    structured_results.append(structured_result)
    results.append(
        {
            "id": test["id"],
            "name": test["name"],
            "input": test["input"],
            "expected": test["expected"],
            "obtained": obtained,
            "structured_output": structured_result.model_dump(),
            "passed": passed,
        }
    )

    print(f"Teste {test['id']:02d} - {test['name']}: {'APROVADO' if passed else 'REPROVADO'}")
    print(f"Entrada: {test['input']}")
    print(f"Esperado: {test['expected']}")
    print(f"Obtido: ferramentas={used_tools}; estado={state.summary()}")
    if reasons:
        print(f"Motivos: {reasons}")
    print()

approved = sum(result["passed"] for result in results)
total = len(results)
accuracy = approved / total * 100

print(f"Taxa de acerto: {approved}/{total} ({accuracy:.1f}%)")


Teste 01 - Análise completa: APROVADO
Entrada: Por que a volta 7 perdeu tempo em relação à volta 5? Identifique onde ocorreu a perda e analise as evidências.
Esperado: Usar compare_laps e analyze_section; target=7, reference=5, setor principal=2.
Obtido: ferramentas=['compare_laps', 'analyze_section']; estado={'target_lap': 7, 'reference_lap': 5, 'compared': True, 'total_time_difference': 0.9000000000000341, 'main_loss_section': 2, 'main_loss': 0.8, 'analyzed_sections': [2], 'evidence_count': 5, 'observation_count': 2}

Teste 02 - Análise de setor: APROVADO
Entrada: Analise o setor 2 da volta 7 comparando com a volta 5.
Esperado: Usar analyze_section; target=7, reference=5, setor=2.
Obtido: ferramentas=['analyze_section']; estado={'target_lap': 7, 'reference_lap': 5, 'compared': False, 'total_time_difference': None, 'main_loss_section': None, 'main_loss': None, 'analyzed_sections': [2], 'evidence_count': 5, 'observation_count': 1}

Teste 03 - Comparação de voltas: APROVADO
Entrada: Com

## Explicação dos testes

1. **Análise completa:** este teste verifica uma tarefa de múltiplas etapas: comparar a volta 7 com a volta 5, encontrar o setor de maior perda e analisar as evidências desse setor. Ele é verificável porque compare_laps calcula a diferença total de tempo e identifica objetivamente o setor 2 como a maior perda. O modelo acertou porque chamou compare_laps e depois analyze_section, registrando target 7, referência 5 e setor 2.

2. **Análise de setor:** este teste verifica se o agente respeita um setor informado diretamente pelo usuário. Ele é verificável porque a entrada fixa volta analisada, volta de referência e setor. O modelo acertou porque chamou analyze_section para a volta 7 contra a volta 5 no setor 2.

3. **Comparação de voltas:** este teste verifica se o agente calcula a diferença entre duas voltas explícitas. Ele é verificável porque a diferença vem dos tempos medidos na telemetria. O modelo acertou porque chamou compare_laps com target 7 e referência 5; ele também chamou analyze_section, que não era obrigatória, mas não prejudicou a validação.

4. **Descoberta da referência:** este teste verifica se o agente consegue descobrir a melhor volta válida antes de comparar. Ele é verificável porque list_laps retorna as voltas válidas e seus tempos, e a melhor volta válida é a volta 6. O modelo acertou porque listou as voltas, comparou a volta 7 com a volta 6 e analisou o setor 2, que foi a maior perda nessa comparação.

5. **Consulta da sessão:** este teste verifica se o agente consulta as voltas disponíveis sem inventar dados. Ele é verificável porque os status das voltas são determinados no pré-processamento. O modelo acertou porque chamou list_laps e não tentou fazer comparação sem necessidade.

6. **Outro setor:** este teste verifica se o agente analisa um setor específico diferente do principal caso de perda. Ele é verificável porque a entrada pede explicitamente o setor 4. O modelo acertou porque chamou analyze_section com volta 7, referência 5 e setor 4.

7. **Referência sem volta definida:** este teste verifica se o agente escolhe uma referência válida para a volta 8 antes da comparação. Ele é verificável porque a melhor referência pode ser determinada pelos tempos das voltas válidas. O modelo acertou porque chamou list_laps, escolheu a volta 6 como referência, comparou com a volta 8 e analisou o setor 4, onde houve a maior perda.

8. **Volta inválida:** este teste verifica o tratamento de uma volta que não pode ser usada em comparação. Ele é verificável porque a volta 1 foi classificada como incomplete no pré-processamento. O modelo acertou porque chamou compare_laps, recebeu a mensagem de invalidez da ferramenta e não registrou uma comparação falsa no estado.

9. **Volta incompleta:** este teste verifica outro caso inválido, agora com a volta 12. Ele é verificável porque a volta 12 não completa a distância mínima da pista e também foi classificada como incomplete. O modelo acertou porque chamou compare_laps, reconheceu a invalidez e não produziu evidências indevidas.

10. **Setor inválido:** este teste verifica a validação de limites da ferramenta de setor. Ele é verificável porque a implementação aceita apenas setores de 1 a 5. O modelo acertou porque chamou analyze_section, recebeu a mensagem de erro para o setor 6 e não marcou nenhum setor como analisado.


## Saída estruturada dos testes

A célula abaixo apresenta uma saída estruturada para todos os casos de teste usando o schema `StructuredTestResult`. Ela não chama o modelo novamente: apenas valida e serializa os resultados já produzidos pelo laço de testes anterior.


In [18]:
structured_payload = [item.model_dump() for item in structured_results]
print(json.dumps(structured_payload, indent=2, ensure_ascii=False))


[
  {
    "test_id": 1,
    "name": "Análise completa",
    "input": "Por que a volta 7 perdeu tempo em relação à volta 5? Identifique onde ocorreu a perda e analise as evidências.",
    "expected": "Usar compare_laps e analyze_section; target=7, reference=5, setor principal=2.",
    "used_tools": [
      "compare_laps",
      "analyze_section"
    ],
    "observed_state": {
      "target_lap": 7,
      "reference_lap": 5,
      "compared": true,
      "total_time_difference": 0.9000000000000341,
      "main_loss_section": 2,
      "main_loss": 0.8,
      "analyzed_sections": [
        2
      ],
      "evidence_count": 5,
      "observation_count": 2
    },
    "final_answer": "Com base na análise da telemetria, identifiquei que a **volta 7 perdeu tempo em relação à volta 5**, com uma diferença total de **0.9 segundos**. A perda ocorreu principalmente no **Setor 2** (de 978.807m a 1957.613m), onde houve uma perda de **0.8 segundos**.\n\n### Evidências quantitativas do Setor 2:\n\n| Mé

## Análise final dos resultados

A execução registrada aprovou os 10 testes, com taxa de acerto de 100.0%. Não houve erro funcional nos casos avaliados: o agente chamou as ferramentas necessárias, manteve o estado coerente com as voltas e setores esperados, identificou corretamente as melhores referências quando elas não foram dadas e tratou entradas inválidas sem inventar resultados.

Os testes 3 e 7 mostram que o agente às vezes chama analyze_section além do mínimo exigido. Isso não foi contado como erro porque a chamada adicional é coerente com o objetivo de explicar a comparação com evidências quantitativas. Nos testes 8 e 9, após receber a mensagem de volta inválida, o agente também chamou list_laps; essa ação extra é aceitável porque ajuda a justificar quais voltas estão disponíveis e não altera indevidamente o estado.

A célula de saída estruturada usa structured_results, produzido pelo laço dos 10 testes, para serializar todos os casos com entrada, expectativa, ferramentas usadas, estado observado, resposta final, aprovação e notas de validação. Ela não refaz o trabalho do modelo e só depende dos resultados já coletados na execução anterior do notebook.

A principal limitação observada foi operacional: a execução dos 10 testes levou cerca de 22 minutos com o modelo local qwen3.5:4b via Ollama, rodano na minha máquina. Esse tempo é esperado para um fluxo com várias chamadas ao modelo e ferramentas em sequência, mesmo com aceleração por GPU/Vulkan. Rodando no Google Colab, esse tempo é bem reduzido.
